In [83]:
import json
import pandas as pd

In [ ]:
dados_artistas = {}

with open("../../data/raw/musicbrainz_data.json", "r", encoding="utf-8") as arquivo:
    dados_artistas = json.load(arquivo)

df_mb = pd.DataFrame.from_dict(dados_artistas, orient="index") 
#ajustando para tratar as chaves externas como as linhas da tabela, não as colunas (padrão)
# index se refere as chaves externas como linhas -> nomes dos artistas

df_mb.head(10)

# count mostra quantos outros artistas semelhantes batem com a nossa busca

,created,count,offset,artists
CMAT,2026-09-21T20:54:11.975Z,14,0,[{'id': '2ac533ad-f3ec-4083-a2d4-6f30393414b3'...
Sufjan Stevens,2026-09-21T20:54:13.790Z,1044,0,[{'id': '01d3c51b-9b98-418a-8d8e-37f6fab59d8c'...
Sabrina Carpenter,2026-09-21T20:54:18.346Z,658,0,[{'id': '1882fe91-cdd9-49c9-9956-8e06a3810bd4'...
Banda Amores,2026-09-21T20:54:20.514Z,1772,0,[{'id': 'a8d00992-f534-47c9-ba76-bd668e8ec508'...
Alceu Valença,2026-09-21T20:54:22.708Z,15,0,[{'id': '7a08d4fb-e5c1-4c39-a99b-1908ca2e83b4'...
Tinashe,2026-09-21T20:54:24.812Z,14,0,[{'id': '396a9525-0994-4542-b480-c4587feeb476'...
Zé Ibarra,2026-09-21T20:54:26.898Z,530,0,[{'id': '0b2665a9-14c0-499a-b9fd-46ed09bbf161'...
Olivia Rodrigo,2026-09-21T20:54:28.990Z,1303,0,[{'id': '6925db17-f35e-42f3-a4eb-84ee6bf5d4b0'...
Pabllo Vittar,2026-09-21T20:53:57.262Z,3,0,[{'id': 'd9550ac2-7bcf-4655-a3fe-24922e254a74'...
Lana Del Rey,2026-09-21T20:54:33.200Z,29385,0,[{'id': 'b7539c32-53e7-4908-bda3-81449c367da6'...


In [85]:
df_mb.shape

(172, 4)

In [86]:
df_mb.columns

Index(['created', 'count', 'offset', 'artists'], dtype='str')

In [87]:
df_mb.drop(columns=['created', 'count', 'offset'], inplace=True)

In [88]:
df_mb.head()

,artists
CMAT,[{'id': '2ac533ad-f3ec-4083-a2d4-6f30393414b3'...
Sufjan Stevens,[{'id': '01d3c51b-9b98-418a-8d8e-37f6fab59d8c'...
Sabrina Carpenter,[{'id': '1882fe91-cdd9-49c9-9956-8e06a3810bd4'...
Banda Amores,[{'id': 'a8d00992-f534-47c9-ba76-bd668e8ec508'...
Alceu Valença,[{'id': '7a08d4fb-e5c1-4c39-a99b-1908ca2e83b4'...


In [ ]:
df_mb['query_name'] = df_mb.index # salvando index para conexão com banco do last fm -> pega o indice estabelecido anteriormente e adiciona a tabela p ligação
df_mb_exploded = df_mb.explode('artists').reset_index(drop=True)
# explode = pega os dicionários aninhados numa lista e transforma em linhas
df_mb_normalizado = pd.json_normalize(df_mb_exploded['artists']).reset_index(drop=True)
#o json_normalize pega esses dícionários e transforma suas prorpiedades em colunas
df_mb_normalizado['query_name'] = df_mb_exploded['query_name'] #passando index para tabela 
df_mb_normalizado.head(10)

,id,type,type-id,score,gender-id,name,sort-name,gender,country,isnis,...,disambiguation,end-area.id,end-area.type,end-area.type-id,end-area.name,end-area.sort-name,end-area.life-span.ended,life-span.end,area.life-span.begin,query_name
0,2ac533ad-f3ec-4083-a2d4-6f30393414b3,Person,b6e035f4-3ce9-331c-97df-83397230b0df,100,93452b5a-a947-30c8-934f-6a4056b151c2,CMAT,CMAT,female,IE,[0000000507072282],...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,CMAT
1,01d3c51b-9b98-418a-8d8e-37f6fab59d8c,Person,b6e035f4-3ce9-331c-97df-83397230b0df,100,36d3d30a-839d-3eda-8cb3-29be4384e4a9,Sufjan Stevens,"Stevens, Sufjan",male,US,[0000000110253464],...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Sufjan Stevens
2,1882fe91-cdd9-49c9-9956-8e06a3810bd4,Person,b6e035f4-3ce9-331c-97df-83397230b0df,100,93452b5a-a947-30c8-934f-6a4056b151c2,Sabrina Carpenter,"Carpenter, Sabrina",female,US,[0000000440350523],...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Sabrina Carpenter
3,a8d00992-f534-47c9-ba76-bd668e8ec508,Group,e431f5f6-b5d2-343d-8b36-72607fffb74b,100,NaN,Amores,Amores,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Banda Amores
4,7a08d4fb-e5c1-4c39-a99b-1908ca2e83b4,Person,b6e035f4-3ce9-331c-97df-83397230b0df,100,36d3d30a-839d-3eda-8cb3-29be4384e4a9,Alceu Valença,"Valença, Alceu",male,BR,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Alceu Valença
5,396a9525-0994-4542-b480-c4587feeb476,Person,b6e035f4-3ce9-331c-97df-83397230b0df,100,93452b5a-a947-30c8-934f-6a4056b151c2,Tinashe,Tinashe,female,US,[000000010834165X],...,American R&B singer,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Tinashe
6,0b2665a9-14c0-499a-b9fd-46ed09bbf161,Person,b6e035f4-3ce9-331c-97df-83397230b0df,100,36d3d30a-839d-3eda-8cb3-29be4384e4a9,Zé Ibarra,"Ibarra, Zé",male,NaN,[0000000493736411],...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Zé Ibarra
7,6925db17-f35e-42f3-a4eb-84ee6bf5d4b0,Person,b6e035f4-3ce9-331c-97df-83397230b0df,100,93452b5a-a947-30c8-934f-6a4056b151c2,Olivia Rodrigo,"Rodrigo, Olivia",female,US,[000000047619573X],...,US singer‐songwriter,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Olivia Rodrigo
8,d9550ac2-7bcf-4655-a3fe-24922e254a74,Person,b6e035f4-3ce9-331c-97df-83397230b0df,100,b239bbd8-aa84-4f11-8bdb-c4159e730ee5,Pabllo Vittar,"Vittar, Pabllo",non-binary,BR,[0000000466937285],...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Pabllo Vittar
9,b7539c32-53e7-4908-bda3-81449c367da6,Person,b6e035f4-3ce9-331c-97df-83397230b0df,100,93452b5a-a947-30c8-934f-6a4056b151c2,Lana Del Rey,"Del Rey, Lana",female,US,"[0000000360454402, 0000000368678910]",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Lana Del Rey


In [90]:
df_mb_normalizado.columns

Index(['id', 'type', 'type-id', 'score', 'gender-id', 'name', 'sort-name',
       'gender', 'country', 'isnis', 'aliases', 'tags', 'area.id', 'area.type',
       'area.type-id', 'area.name', 'area.sort-name', 'area.life-span.ended',
       'begin-area.id', 'begin-area.type', 'begin-area.type-id',
       'begin-area.name', 'begin-area.sort-name', 'begin-area.life-span.ended',
       'life-span.begin', 'life-span.ended', 'ipis', 'disambiguation',
       'end-area.id', 'end-area.type', 'end-area.type-id', 'end-area.name',
       'end-area.sort-name', 'end-area.life-span.ended', 'life-span.end',
       'area.life-span.begin', 'query_name'],
      dtype='str')

In [91]:
df_mb_normalizado.drop(columns=['score','type-id', 'gender-id', 'sort-name', 'isnis', 'area.id', 'area.type-id', 'area.sort-name', 
                                'area.life-span.ended', 'begin-area.id', 'begin-area.type', 'begin-area.type-id',
                                'begin-area.name', 'begin-area.sort-name', 'begin-area.life-span.ended',
                                'life-span.begin', 'life-span.ended', 'ipis','disambiguation', 'end-area.id', 'end-area.type', 'end-area.type-id', 'end-area.name',
                                'end-area.sort-name', 'end-area.life-span.ended', 'life-span.end',
                                'area.life-span.begin'], inplace=True)

df_mb_normalizado.head(10)

,id,type,name,gender,country,aliases,tags,area.type,area.name,query_name
0,2ac533ad-f3ec-4083-a2d4-6f30393414b3,Person,CMAT,female,IE,"[{'sort-name': 'Thompson, Ciara Mary-Alice', '...","[{'count': 1, 'name': 'pop'}, {'count': 1, 'na...",Country,Ireland,CMAT
1,01d3c51b-9b98-418a-8d8e-37f6fab59d8c,Person,Sufjan Stevens,male,US,"[{'sort-name': 'Stevens', 'name': 'Stevens', '...","[{'count': 0, 'name': 'alternative rock'}, {'c...",Country,United States,Sufjan Stevens
2,1882fe91-cdd9-49c9-9956-8e06a3810bd4,Person,Sabrina Carpenter,female,US,"[{'sort-name': 'Carpenter, Sabrina', 'type-id'...","[{'count': 9, 'name': 'pop'}, {'count': 1, 'na...",Country,United States,Sabrina Carpenter
3,a8d00992-f534-47c9-ba76-bd668e8ec508,Group,Amores,NaN,NaN,NaN,NaN,NaN,NaN,Banda Amores
4,7a08d4fb-e5c1-4c39-a99b-1908ca2e83b4,Person,Alceu Valença,male,BR,"[{'sort-name': 'Valença, Alceu Paiva', 'type-i...","[{'count': 2, 'name': 'pop'}, {'count': 1, 'na...",Country,Brazil,Alceu Valença
5,396a9525-0994-4542-b480-c4587feeb476,Person,Tinashe,female,US,"[{'sort-name': 'Jorgenson Kachingwe, Tinashe',...","[{'count': 2, 'name': 'pop'}, {'count': 2, 'na...",Country,United States,Tinashe
6,0b2665a9-14c0-499a-b9fd-46ed09bbf161,Person,Zé Ibarra,male,NaN,NaN,NaN,City,Rio de Janeiro,Zé Ibarra
7,6925db17-f35e-42f3-a4eb-84ee6bf5d4b0,Person,Olivia Rodrigo,female,US,"[{'sort-name': '奥利维亚·罗德里戈', 'type-id': '894afb...","[{'count': 1, 'name': 'rock'}, {'count': 14, '...",Country,United States,Olivia Rodrigo
8,d9550ac2-7bcf-4655-a3fe-24922e254a74,Person,Pabllo Vittar,non-binary,BR,"[{'sort-name': 'Silva, Phabullo Rodrigues da',...","[{'count': 3, 'name': 'drag queen'}, {'count':...",Country,Brazil,Pabllo Vittar
9,b7539c32-53e7-4908-bda3-81449c367da6,Person,Lana Del Rey,female,US,"[{'sort-name': 'Del Rey, Lana', 'type-id': '89...","[{'count': 1, 'name': 'rock'}, {'count': 5, 'n...",Country,United States,Lana Del Rey


In [92]:
df_mb_normalizado['aliases'].head(10)

0    [{'sort-name': 'Thompson, Ciara Mary-Alice', '...
1    [{'sort-name': 'Stevens', 'name': 'Stevens', '...
2    [{'sort-name': 'Carpenter, Sabrina', 'type-id'...
3                                                  NaN
4    [{'sort-name': 'Valença, Alceu Paiva', 'type-i...
5    [{'sort-name': 'Jorgenson Kachingwe, Tinashe',...
6                                                  NaN
7    [{'sort-name': '奥利维亚·罗德里戈', 'type-id': '894afb...
8    [{'sort-name': 'Silva, Phabullo Rodrigues da',...
9    [{'sort-name': 'Del Rey, Lana', 'type-id': '89...
Name: aliases, dtype: object

In [93]:
df_mb_normalizado['legal_name'] = df_mb_normalizado['aliases'].apply(lambda lista: next(
    (a['name'] for a in lista if a.get('type') == 'Legal name'), 'unregistered') if isinstance(lista, list) else 'unregistered'
    )

# next(retorna variavel, se encontrar -> for, se não lança esse nome no lugar da exceção)

In [94]:
df_mb_normalizado.head(5)

,id,type,name,gender,country,aliases,tags,area.type,area.name,query_name,legal_name
0,2ac533ad-f3ec-4083-a2d4-6f30393414b3,Person,CMAT,female,IE,"[{'sort-name': 'Thompson, Ciara Mary-Alice', '...","[{'count': 1, 'name': 'pop'}, {'count': 1, 'na...",Country,Ireland,CMAT,Ciara Mary-Alice Thompson
1,01d3c51b-9b98-418a-8d8e-37f6fab59d8c,Person,Sufjan Stevens,male,US,"[{'sort-name': 'Stevens', 'name': 'Stevens', '...","[{'count': 0, 'name': 'alternative rock'}, {'c...",Country,United States,Sufjan Stevens,unregistered
2,1882fe91-cdd9-49c9-9956-8e06a3810bd4,Person,Sabrina Carpenter,female,US,"[{'sort-name': 'Carpenter, Sabrina', 'type-id'...","[{'count': 9, 'name': 'pop'}, {'count': 1, 'na...",Country,United States,Sabrina Carpenter,Sabrina Annlynn Carpenter
3,a8d00992-f534-47c9-ba76-bd668e8ec508,Group,Amores,NaN,NaN,NaN,NaN,NaN,NaN,Banda Amores,unregistered
4,7a08d4fb-e5c1-4c39-a99b-1908ca2e83b4,Person,Alceu Valença,male,BR,"[{'sort-name': 'Valença, Alceu Paiva', 'type-i...","[{'count': 2, 'name': 'pop'}, {'count': 1, 'na...",Country,Brazil,Alceu Valença,Alceu Paiva Valença


In [95]:
# Primeiro método de tratamento de nomes que tentei
# df_mb_normalizado['legal_name'] = df_mb_normalizado['aliases'].str[0].str['name'].fillna('unregistered')
# # tratando dados nulos do nome legal
# # aliases é uma lista de dicionários, com .str consigo acessar cada elemento dentro do dicionário
# # se for NaN, substitui por um texto fixo
df_mb_normalizado['type'] = df_mb_normalizado['aliases'].str[0].str['type'].fillna('unregisterd')
df_mb_normalizado.drop(columns=['aliases'], inplace=True)
df_mb_normalizado.head(10)

,id,type,name,gender,country,tags,area.type,area.name,query_name,legal_name
0,2ac533ad-f3ec-4083-a2d4-6f30393414b3,Legal name,CMAT,female,IE,"[{'count': 1, 'name': 'pop'}, {'count': 1, 'na...",Country,Ireland,CMAT,Ciara Mary-Alice Thompson
1,01d3c51b-9b98-418a-8d8e-37f6fab59d8c,unregisterd,Sufjan Stevens,male,US,"[{'count': 0, 'name': 'alternative rock'}, {'c...",Country,United States,Sufjan Stevens,unregistered
2,1882fe91-cdd9-49c9-9956-8e06a3810bd4,Artist name,Sabrina Carpenter,female,US,"[{'count': 9, 'name': 'pop'}, {'count': 1, 'na...",Country,United States,Sabrina Carpenter,Sabrina Annlynn Carpenter
3,a8d00992-f534-47c9-ba76-bd668e8ec508,unregisterd,Amores,NaN,NaN,NaN,NaN,NaN,Banda Amores,unregistered
4,7a08d4fb-e5c1-4c39-a99b-1908ca2e83b4,Legal name,Alceu Valença,male,BR,"[{'count': 2, 'name': 'pop'}, {'count': 1, 'na...",Country,Brazil,Alceu Valença,Alceu Paiva Valença
5,396a9525-0994-4542-b480-c4587feeb476,Legal name,Tinashe,female,US,"[{'count': 2, 'name': 'pop'}, {'count': 2, 'na...",Country,United States,Tinashe,Tinashe Jorgenson Kachingwe
6,0b2665a9-14c0-499a-b9fd-46ed09bbf161,unregisterd,Zé Ibarra,male,NaN,NaN,City,Rio de Janeiro,Zé Ibarra,unregistered
7,6925db17-f35e-42f3-a4eb-84ee6bf5d4b0,Artist name,Olivia Rodrigo,female,US,"[{'count': 1, 'name': 'rock'}, {'count': 14, '...",Country,United States,Olivia Rodrigo,Olivia Isabel Rodrigo
8,d9550ac2-7bcf-4655-a3fe-24922e254a74,Legal name,Pabllo Vittar,non-binary,BR,"[{'count': 3, 'name': 'drag queen'}, {'count':...",Country,Brazil,Pabllo Vittar,Phabullo Rodrigues da Silva
9,b7539c32-53e7-4908-bda3-81449c367da6,Artist name,Lana Del Rey,female,US,"[{'count': 1, 'name': 'rock'}, {'count': 5, 'n...",Country,United States,Lana Del Rey,Elizabeth Woolridge Grant


In [96]:
df_mb_normalizado.isna()

,id,type,name,gender,country,tags,area.type,area.name,query_name,legal_name
0,False,False,False,False,False,False,False,False,False,False
1,False,False,False,False,False,False,False,False,False,False
2,False,False,False,False,False,False,False,False,False,False
3,False,False,False,True,True,True,True,True,False,False
4,False,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...
167,False,False,False,False,False,False,False,False,False,False
168,False,False,False,True,False,False,False,False,False,False
169,False,False,False,False,False,False,False,False,False,False
170,False,False,False,True,False,False,False,False,False,False


In [97]:
df_mb_normalizado['gender'] = df_mb_normalizado['gender'].fillna('unregisterd')
df_mb_normalizado['area.type'] = df_mb_normalizado['area.type'].fillna('unregisterd')
df_mb_normalizado['area.name'] = df_mb_normalizado['area.name'].fillna('unregisterd')
df_mb_normalizado['country'] = df_mb_normalizado['country'].fillna('unregisterd')
df_mb_normalizado['music_genre'] = df_mb_normalizado['tags'].str[0].str['name'].fillna('unregisterd')

In [98]:
df_mb_normalizado['tags']

0      [{'count': 1, 'name': 'pop'}, {'count': 1, 'na...
1      [{'count': 0, 'name': 'alternative rock'}, {'c...
2      [{'count': 9, 'name': 'pop'}, {'count': 1, 'na...
3                                                    NaN
4      [{'count': 2, 'name': 'pop'}, {'count': 1, 'na...
                             ...                        
167    [{'count': 1, 'name': 'pop'}, {'count': 1, 'na...
168    [{'count': 1, 'name': 'rock'}, {'count': 2, 'n...
169    [{'count': 1, 'name': 'brazilian'}, {'count': ...
170    [{'count': 2, 'name': 'pop'}, {'count': 3, 'na...
171    [{'count': 1, 'name': 'girl group'}, {'count':...
Name: tags, Length: 172, dtype: object

In [99]:
df_mb_normalizado.isna()

,id,type,name,gender,country,tags,area.type,area.name,query_name,legal_name,music_genre
0,False,False,False,False,False,False,False,False,False,False,False
1,False,False,False,False,False,False,False,False,False,False,False
2,False,False,False,False,False,False,False,False,False,False,False
3,False,False,False,False,False,True,False,False,False,False,False
4,False,False,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...
167,False,False,False,False,False,False,False,False,False,False,False
168,False,False,False,False,False,False,False,False,False,False,False
169,False,False,False,False,False,False,False,False,False,False,False
170,False,False,False,False,False,False,False,False,False,False,False


In [100]:
df_mb_normalizado.head(10)

,id,type,name,gender,country,tags,area.type,area.name,query_name,legal_name,music_genre
0,2ac533ad-f3ec-4083-a2d4-6f30393414b3,Legal name,CMAT,female,IE,"[{'count': 1, 'name': 'pop'}, {'count': 1, 'na...",Country,Ireland,CMAT,Ciara Mary-Alice Thompson,pop
1,01d3c51b-9b98-418a-8d8e-37f6fab59d8c,unregisterd,Sufjan Stevens,male,US,"[{'count': 0, 'name': 'alternative rock'}, {'c...",Country,United States,Sufjan Stevens,unregistered,alternative rock
2,1882fe91-cdd9-49c9-9956-8e06a3810bd4,Artist name,Sabrina Carpenter,female,US,"[{'count': 9, 'name': 'pop'}, {'count': 1, 'na...",Country,United States,Sabrina Carpenter,Sabrina Annlynn Carpenter,pop
3,a8d00992-f534-47c9-ba76-bd668e8ec508,unregisterd,Amores,unregisterd,unregisterd,NaN,unregisterd,unregisterd,Banda Amores,unregistered,unregisterd
4,7a08d4fb-e5c1-4c39-a99b-1908ca2e83b4,Legal name,Alceu Valença,male,BR,"[{'count': 2, 'name': 'pop'}, {'count': 1, 'na...",Country,Brazil,Alceu Valença,Alceu Paiva Valença,pop
5,396a9525-0994-4542-b480-c4587feeb476,Legal name,Tinashe,female,US,"[{'count': 2, 'name': 'pop'}, {'count': 2, 'na...",Country,United States,Tinashe,Tinashe Jorgenson Kachingwe,pop
6,0b2665a9-14c0-499a-b9fd-46ed09bbf161,unregisterd,Zé Ibarra,male,unregisterd,NaN,City,Rio de Janeiro,Zé Ibarra,unregistered,unregisterd
7,6925db17-f35e-42f3-a4eb-84ee6bf5d4b0,Artist name,Olivia Rodrigo,female,US,"[{'count': 1, 'name': 'rock'}, {'count': 14, '...",Country,United States,Olivia Rodrigo,Olivia Isabel Rodrigo,rock
8,d9550ac2-7bcf-4655-a3fe-24922e254a74,Legal name,Pabllo Vittar,non-binary,BR,"[{'count': 3, 'name': 'drag queen'}, {'count':...",Country,Brazil,Pabllo Vittar,Phabullo Rodrigues da Silva,drag queen
9,b7539c32-53e7-4908-bda3-81449c367da6,Artist name,Lana Del Rey,female,US,"[{'count': 1, 'name': 'rock'}, {'count': 5, 'n...",Country,United States,Lana Del Rey,Elizabeth Woolridge Grant,rock


In [101]:
df_mb_normalizado.drop(columns=['tags'], inplace=True)

In [102]:
# df_mb['music_genre'] = df_mb_normalizado['tags'].apply(lambda lista : max(lista, key= lambda tag: tag['count'])['name'])
# # tags é cada dicionário dentro de lista, que é atualmente uma lista de dicionários
# # usando lambda dentro do max, pegamos o nome de cada gênero que possui maior número de votações

# # count aqui não é a quantidade de vezes que aparece mais, mas que o music Brainz os usuários votam no estilo musical que mais combina com o artista

# df_mb.head(10)

In [103]:
df_mb_normalizado[['gender', 'country','area.type', 'area.name']] = df_mb_normalizado[['gender', 'country','area.type', 'area.name']].fillna('unregistered')

In [104]:
df_mb_normalizado.head(20)

,id,type,name,gender,country,area.type,area.name,query_name,legal_name,music_genre
0,2ac533ad-f3ec-4083-a2d4-6f30393414b3,Legal name,CMAT,female,IE,Country,Ireland,CMAT,Ciara Mary-Alice Thompson,pop
1,01d3c51b-9b98-418a-8d8e-37f6fab59d8c,unregisterd,Sufjan Stevens,male,US,Country,United States,Sufjan Stevens,unregistered,alternative rock
2,1882fe91-cdd9-49c9-9956-8e06a3810bd4,Artist name,Sabrina Carpenter,female,US,Country,United States,Sabrina Carpenter,Sabrina Annlynn Carpenter,pop
3,a8d00992-f534-47c9-ba76-bd668e8ec508,unregisterd,Amores,unregisterd,unregisterd,unregisterd,unregisterd,Banda Amores,unregistered,unregisterd
4,7a08d4fb-e5c1-4c39-a99b-1908ca2e83b4,Legal name,Alceu Valença,male,BR,Country,Brazil,Alceu Valença,Alceu Paiva Valença,pop
5,396a9525-0994-4542-b480-c4587feeb476,Legal name,Tinashe,female,US,Country,United States,Tinashe,Tinashe Jorgenson Kachingwe,pop
6,0b2665a9-14c0-499a-b9fd-46ed09bbf161,unregisterd,Zé Ibarra,male,unregisterd,City,Rio de Janeiro,Zé Ibarra,unregistered,unregisterd
7,6925db17-f35e-42f3-a4eb-84ee6bf5d4b0,Artist name,Olivia Rodrigo,female,US,Country,United States,Olivia Rodrigo,Olivia Isabel Rodrigo,rock
8,d9550ac2-7bcf-4655-a3fe-24922e254a74,Legal name,Pabllo Vittar,non-binary,BR,Country,Brazil,Pabllo Vittar,Phabullo Rodrigues da Silva,drag queen
9,b7539c32-53e7-4908-bda3-81449c367da6,Artist name,Lana Del Rey,female,US,Country,United States,Lana Del Rey,Elizabeth Woolridge Grant,rock


In [105]:
df_mb_normalizado.isna()

,id,type,name,gender,country,area.type,area.name,query_name,legal_name,music_genre
0,False,False,False,False,False,False,False,False,False,False
1,False,False,False,False,False,False,False,False,False,False
2,False,False,False,False,False,False,False,False,False,False
3,False,False,False,False,False,False,False,False,False,False
4,False,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...
167,False,False,False,False,False,False,False,False,False,False
168,False,False,False,False,False,False,False,False,False,False
169,False,False,False,False,False,False,False,False,False,False
170,False,False,False,False,False,False,False,False,False,False


In [106]:
df_mb_normalizado.head(30)

,id,type,name,gender,country,area.type,area.name,query_name,legal_name,music_genre
0,2ac533ad-f3ec-4083-a2d4-6f30393414b3,Legal name,CMAT,female,IE,Country,Ireland,CMAT,Ciara Mary-Alice Thompson,pop
1,01d3c51b-9b98-418a-8d8e-37f6fab59d8c,unregisterd,Sufjan Stevens,male,US,Country,United States,Sufjan Stevens,unregistered,alternative rock
2,1882fe91-cdd9-49c9-9956-8e06a3810bd4,Artist name,Sabrina Carpenter,female,US,Country,United States,Sabrina Carpenter,Sabrina Annlynn Carpenter,pop
3,a8d00992-f534-47c9-ba76-bd668e8ec508,unregisterd,Amores,unregisterd,unregisterd,unregisterd,unregisterd,Banda Amores,unregistered,unregisterd
4,7a08d4fb-e5c1-4c39-a99b-1908ca2e83b4,Legal name,Alceu Valença,male,BR,Country,Brazil,Alceu Valença,Alceu Paiva Valença,pop
5,396a9525-0994-4542-b480-c4587feeb476,Legal name,Tinashe,female,US,Country,United States,Tinashe,Tinashe Jorgenson Kachingwe,pop
6,0b2665a9-14c0-499a-b9fd-46ed09bbf161,unregisterd,Zé Ibarra,male,unregisterd,City,Rio de Janeiro,Zé Ibarra,unregistered,unregisterd
7,6925db17-f35e-42f3-a4eb-84ee6bf5d4b0,Artist name,Olivia Rodrigo,female,US,Country,United States,Olivia Rodrigo,Olivia Isabel Rodrigo,rock
8,d9550ac2-7bcf-4655-a3fe-24922e254a74,Legal name,Pabllo Vittar,non-binary,BR,Country,Brazil,Pabllo Vittar,Phabullo Rodrigues da Silva,drag queen
9,b7539c32-53e7-4908-bda3-81449c367da6,Artist name,Lana Del Rey,female,US,Country,United States,Lana Del Rey,Elizabeth Woolridge Grant,rock


In [107]:
df_mb_normalizado.isna().sum()

id             0
type           0
name           0
gender         0
country        0
area.type      0
area.name      0
query_name     0
legal_name     0
music_genre    0
dtype: int64

In [108]:
df_mb_normalizado.to_csv('../../data/processed/musicBrainz_transformed.csv', index=False)

In [109]:
print(df_mb_normalizado.index[:10])

RangeIndex(start=0, stop=10, step=1)
